# EDA: Salud mental en el sector tecnológico

## Preguntas de partida

### Pregunta 1: ¿Quién busca tratamiento?
- **Columnas:** `treatment` según `Gender`, `Age` (en tramos), `family_history` y `Country`.
- **Hipótesis:** tener antecedentes familiares aumenta la probabilidad de buscar tratamiento, y las mujeres lo buscan más que los hombres.
- **Gráfico:** barras con el porcentaje de `treatment` en cada grupo.

### Pregunta 2: ¿Lo que ofrece la empresa cambia algo?
- **Columnas:** `treatment` según `benefits`, `care_options`, `wellness_program` y `seek_help`.
- **Hipótesis:** en las empresas que ofrecen más apoyo, más gente busca tratamiento.
- **Gráfico:** barras comparando el porcentaje de `treatment` con y sin cada tipo de apoyo.
- **Precaución:** una relación no prueba que el apoyo cause el tratamiento.

### Pregunta 3: ¿Quién tiene miedo de hablar de salud mental en el trabajo?
- **Columnas:** `mental_health_consequence`, `supervisor` y `coworkers` según `no_employees`, `remote_work` y `tech_company`.
- **Hipótesis:** en las empresas grandes hay más miedo a las consecuencias, y el teletrabajo cambia la confianza con el supervisor.
- **Gráfico:** barras apiladas en porcentaje por tamaño de empresa.




In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

Matplotlib is building the font cache; this may take a moment.


In [2]:
df = pd.read_csv('data/survey.csv', na_values=["NA"])

In [3]:
df.head()

,Timestamp,Age,Gender,Country,state,self_employed,family_history,treatment,work_interfere,no_employees,...,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence,comments
0,2014-08-27 11:29:31,37,Female,United States,IL,NaN,No,Yes,Often,6-25,...,Somewhat easy,No,No,Some of them,Yes,No,Maybe,Yes,No,NaN
1,2014-08-27 11:29:37,44,M,United States,IN,NaN,No,No,Rarely,More than 1000,...,Don't know,Maybe,No,No,No,No,No,Don't know,No,NaN
2,2014-08-27 11:29:44,32,Male,Canada,NaN,NaN,No,No,Rarely,6-25,...,Somewhat difficult,No,No,Yes,Yes,Yes,Yes,No,No,NaN
3,2014-08-27 11:29:46,31,Male,United Kingdom,NaN,NaN,Yes,Yes,Often,26-100,...,Somewhat difficult,Yes,Yes,Some of them,No,Maybe,Maybe,No,Yes,NaN
4,2014-08-27 11:30:22,31,Male,United States,TX,NaN,No,No,Never,100-500,...,Don't know,No,No,Some of them,Yes,Yes,Yes,Don't know,No,NaN


## Primer vistazo a los datos

- Estructura: la tabla tiene 27 columnas, y casi todas son categóricas (Yes / No / Maybe).
- Valores vacíos: hay valores vacíos (`NaN`), sobre todo en `state`, `self_employed` y `comments`.
- Género: `Gender` está escrito de varias formas (`Female`, `M`, `Male`), así que habrá que limpiarlo.

In [4]:
df.shape

(1259, 27)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1259 entries, 0 to 1258
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   Timestamp                  1259 non-null   str  
 1   Age                        1259 non-null   int64
 2   Gender                     1259 non-null   str  
 3   Country                    1259 non-null   str  
 4   state                      744 non-null    str  
 5   self_employed              1241 non-null   str  
 6   family_history             1259 non-null   str  
 7   treatment                  1259 non-null   str  
 8   work_interfere             995 non-null    str  
 9   no_employees               1259 non-null   str  
 10  remote_work                1259 non-null   str  
 11  tech_company               1259 non-null   str  
 12  benefits                   1259 non-null   str  
 13  care_options               1259 non-null   str  
 14  wellness_program           1259 non

In [6]:
df.isna().sum()

Timestamp                       0
Age                             0
Gender                          0
Country                         0
state                         515
self_employed                  18
family_history                  0
treatment                       0
work_interfere                264
no_employees                    0
remote_work                     0
tech_company                    0
benefits                        0
care_options                    0
wellness_program                0
seek_help                       0
anonymity                       0
leave                           0
mental_health_consequence       0
phys_health_consequence         0
coworkers                       0
supervisor                      0
mental_health_interview         0
phys_health_interview           0
mental_vs_physical              0
obs_consequence                 0
comments                     1095
dtype: int64

El dataset tiene 1259 filas y 27 columnas. Casi todas son categóricas.
Cuatro columnas tienen valores vacíos: `state`, `self_employed`, `work_interfere` y `comments`. La que más tiene es `comments`.

In [7]:
df.describe()

,Age
count,1.259000e+03
mean,7.942815e+07
std,2.818299e+09
min,-1.726000e+03
25%,2.700000e+01
50%,3.100000e+01
75%,3.600000e+01
max,1.000000e+11


In [8]:
df["Age"].sort_values().head(10)

715    -1726
143      -29
1127      -1
734        5
989        8
1090      11
282       18
287       18
43        18
482       18
Name: Age, dtype: int64

In [9]:
df["Age"].sort_values().tail(10)

466              57
471              58
1236             60
297              60
952              61
520              62
560              65
1182             72
364             329
390     99999999999
Name: Age, dtype: int64

Aquí podemos ver en la columna `Age` que hay valores imposibles, como los negativos (-1726, -29, -1), 329 o 99999999999.
Además, los valores 5, 8 y 11 son improbables en una encuesta de trabajo. El 72 es raro, pero posible.
Estos valores hacen que la media no sea fiable  mientras que la mediana (31 años) sí es realista. 

In [11]:
df.describe(include="str")

,Timestamp,Gender,Country,state,self_employed,family_history,treatment,work_interfere,no_employees,remote_work,...,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence,comments
count,1259,1259,1259,744,1241,1259,1259,995,1259,1259,...,1259,1259,1259,1259,1259,1259,1259,1259,1259,164
unique,1246,49,48,45,2,2,2,4,6,2,...,5,3,3,3,3,3,3,3,2,160
top,2014-08-27 12:31:41,Male,United States,CA,No,No,Yes,Sometimes,6-25,No,...,Don't know,No,No,Some of them,Yes,No,Maybe,Don't know,No,* Small family business - YMMV.
freq,2,615,751,138,1095,767,637,465,290,883,...,563,490,925,774,516,1008,557,576,1075,5


- podemos ver que Gender tiene 49 valores distinctos habra que limpiarlo para agruparlo en categorías más amplias.
- Casi 60% de las respuestas son de stados unidos. 

In [12]:
df.columns

Index(['Timestamp', 'Age', 'Gender', 'Country', 'state', 'self_employed',
       'family_history', 'treatment', 'work_interfere', 'no_employees',
       'remote_work', 'tech_company', 'benefits', 'care_options',
       'wellness_program', 'seek_help', 'anonymity', 'leave',
       'mental_health_consequence', 'phys_health_consequence', 'coworkers',
       'supervisor', 'mental_health_interview', 'phys_health_interview',
       'mental_vs_physical', 'obs_consequence', 'comments'],
      dtype='str')

las 27 columnas son divididas entre el perfil del encuestado , la salud mental , y el entorno laboral/apoyo empresarial.